In [0]:
# ==========================================
# Healthcare Lakehouse Configuration
# ==========================================

CATALOG = "workspace"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
AUDIT_SCHEMA = "audit"

RAW_DATA_PATH = "/Volumes/workspace/default/raw_data"

print("Project Configuration Loaded Successfully!")

Project Configuration Loaded Successfully!


In [0]:
print(f"Catalog      : {CATALOG}")
print(f"Raw Data Path: {RAW_DATA_PATH}")

spark.sql("SHOW SCHEMAS IN workspace").show()

Catalog      : workspace
Raw Data Path: /Volumes/workspace/default/raw_data
+------------------+
|      databaseName|
+------------------+
|             audit|
|            bronze|
|           default|
|              gold|
|information_schema|
|          metadata|
|            silver|
+------------------+



In [0]:
display(dbutils.fs.ls(RAW_DATA_PATH))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/raw_data/appointments.csv,appointments.csv,10907,1782973100000
dbfs:/Volumes/workspace/default/raw_data/billing.csv,billing.csv,10018,1782973099000
dbfs:/Volumes/workspace/default/raw_data/doctors.csv,doctors.csv,962,1782973100000
dbfs:/Volumes/workspace/default/raw_data/patients.csv,patients.csv,5626,1782973099000
dbfs:/Volumes/workspace/default/raw_data/treatments.csv,treatments.csv,11072,1782973099000


In [0]:
from pyspark.sql.types import *

metadata_schema = StructType([
    StructField("source_name", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("primary_key", StringType(), False),
    StructField("active_flag", StringType(), False)
])

In [0]:
metadata = [
    ("patients", "patients.csv", "patients", "patient_id", "Y"),
    ("doctors", "doctors.csv", "doctors", "doctor_id", "Y"),
    ("appointments", "appointments.csv", "appointments", "appointment_id", "Y"),
    ("treatments", "treatments.csv", "treatments", "treatment_id", "Y"),
    ("billing", "billing.csv", "billing", "bill_id", "Y")
]

metadata_df = spark.createDataFrame(metadata, metadata_schema)

display(metadata_df)

source_name,file_name,target_table,primary_key,active_flag
patients,patients.csv,patients,patient_id,Y
doctors,doctors.csv,doctors,doctor_id,Y
appointments,appointments.csv,appointments,appointment_id,Y
treatments,treatments.csv,treatments,treatment_id,Y
billing,billing.csv,billing,bill_id,Y


In [0]:
metadata_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.{METADATA_SCHEMA}.config")

In [0]:
spark.sql(f"SELECT * FROM {CATALOG}.{METADATA_SCHEMA}.config").show()

+------------+----------------+------------+--------------+-----------+
| source_name|       file_name|target_table|   primary_key|active_flag|
+------------+----------------+------------+--------------+-----------+
|    patients|    patients.csv|    patients|    patient_id|          Y|
|     doctors|     doctors.csv|     doctors|     doctor_id|          Y|
|appointments|appointments.csv|appointments|appointment_id|          Y|
|  treatments|  treatments.csv|  treatments|  treatment_id|          Y|
|     billing|     billing.csv|     billing|       bill_id|          Y|
+------------+----------------+------------+--------------+-----------+

